# Fire Detection Training in Google Colab

This notebook installs the required packages, downloads a public fire-and-smoke dataset from Roboflow in YOLOv8 format, trains a `YOLOv8s` model on GPU, prints validation metrics, and copies the best weights to Google Drive.


## 1. Install Dependencies

This section installs the `ultralytics` and `roboflow` packages needed for dataset download, training, and evaluation.


In [ ]:
%pip install -q ultralytics roboflow


## 2. Download the Roboflow Dataset

This section connects to Roboflow using your API key and downloads the public Middle East Tech University fire-and-smoke dataset (version `2`) in YOLOv8 format. This dataset contains `15,345` labeled images, which comfortably exceeds the 1,000-image requirement.


In [ ]:
from pathlib import Path

from roboflow import Roboflow

ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"
WORKSPACE = "middle-east-tech-university"
PROJECT = "fire-and-smoke-detection-hiwia"
VERSION = 2

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

data_yaml = Path(dataset.location) / "data.yaml"
print(f"Dataset downloaded to: {dataset.location}")
print(f"Using data config: {data_yaml}")


## 3. Train YOLOv8s on GPU

This section verifies that a GPU runtime is available in Colab and trains `yolov8s.pt` for `50` epochs at `640` image size on `device=0`.


In [ ]:
import torch
from ultralytics import YOLO

assert torch.cuda.is_available(), "GPU not detected. In Colab, select Runtime -> Change runtime type -> GPU before running training."

model = YOLO("yolov8s.pt")
_ = model.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=640,
    device=0,
    project="runs/detect",
    name="train",
    exist_ok=True,
)

print("Training completed. Best weights should be saved to /content/runs/detect/train/weights/best.pt")


## 4. Validate the Best Checkpoint

This section loads the trained `best.pt` checkpoint, runs validation on the dataset, and prints `mAP50`, `precision`, and `recall`.


In [ ]:
from pathlib import Path

from ultralytics import YOLO

best_model_path = Path("/content/runs/detect/train/weights/best.pt")
assert best_model_path.exists(), "best.pt was not found. Make sure training finished successfully before running validation."
best_model = YOLO(str(best_model_path))
metrics = best_model.val(data=str(data_yaml), imgsz=640, device=0, split="val")

print(f"mAP50: {metrics.box.map50:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")


## 5. Mount Google Drive and Save the Best Weights

This section mounts Google Drive, creates the destination folder if needed, and copies `runs/detect/train/weights/best.pt` to `/content/drive/MyDrive/fire_detection/best.pt`.


In [ ]:
from pathlib import Path

from google.colab import drive
import shutil

drive.mount("/content/drive")

source_path = Path("/content/runs/detect/train/weights/best.pt")
assert source_path.exists(), "best.pt was not found. Make sure training finished successfully before copying the weights."
destination_dir = Path("/content/drive/MyDrive/fire_detection")
destination_dir.mkdir(parents=True, exist_ok=True)
destination_path = destination_dir / "best.pt"

shutil.copy2(source_path, destination_path)
print(f"Copied best weights to: {destination_path}")
